In [6]:
import time
from stable_baselines3 import PPO

from spotmicro.env.spotmicro_env import SpotmicroEnv
from spotmicro.physics.factory import create_backend
from spotmicro.devices.fixed_controller import FixedController
from spotmicro.tools.config import Config
from reward_function import reward_function, RewardState

In [7]:
policy = "jumpPB"

cfg = Config()
dev = FixedController("still")
backend = create_backend("mujoco", use_gui=True)
env = SpotmicroEnv(
    backend,
    dev,
    cfg,
    reward_function,
    RewardState(),
    use_gui=True
)
obs, _ = env.reset()

# === Load model ===
model = PPO.load(f"ppo_{policy}", device = 'cpu')
#model = PPO.load(f"{policy}_checkpoints/ppo_{policy}_3000000_steps")
base_steps = env.num_steps

t0 = time.time()
for _ in range(3001):
    action, _ = model.predict(obs, deterministic=True)
    #action = np.array([j.from_position_to_action(hp) for j, hp in zip(env.agent.motor_joints, env.agent.homing_positions)])
    obs, reward, terminated, truncated, info = env.step(action)
    
    time.sleep(1/60.)
    if terminated or truncated:
        print("Terminated")
        env.plot_reward_components()  # plot per episode
        obs, _ = env.reset()
        print(f"Num steps: {env.num_steps - base_steps}")
        break
    
t1 = time.time()
print(f"Elapsed real time: {t1-t0}")

env.close()

Terminated
Num steps: 68
Elapsed real time: 1.9884533882141113
